# RAG Databricks Bluetab - Configuración y Utilidades

Este notebook contiene variables de configuración y funciones utilitarias compartidas para el pipeline RAG en Databricks.

## Características
- Gestión centralizada de configuración
- Widgets para personalización
- Funciones utilitarias compartidas
- Gestión de experimentos MLflow

## Uso
Ejecuta este notebook primero o impórtalo en otros notebooks para establecer parámetros comunes.

In [0]:
# Importar librerías requeridas
import mlflow
import os
from datetime import datetime

## Widgets de configuración
Crea widgets para definir parámetros principales del pipeline y tablas.

In [0]:
# Crear widgets de configuración
# Core Configuration
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")

# Table Configuration
dbutils.widgets.text("docs_text_table", "docs_text", "Documents Text Table")
dbutils.widgets.text("docs_track_table", "docs_track", "Documents Tracking Table")
dbutils.widgets.text("embeddings_table", "docs_text_embeddings", "Embeddings Table")

# Volume Configuration
dbutils.widgets.text("pdf_volume_path", "/Volumes/bluetab/rag/pdf_vol", "PDF Volume Path")

# Model Configuration
dbutils.widgets.text("embedding_model_name", "simple_embedding_model_bluetab", "Embedding Model Name")
dbutils.widgets.text("llm_model_name", "flan_t5_base_model", "LLM Model Name")
dbutils.widgets.text("rag_model_name", "appliance_chatbot_model", "RAG Model Name")

# Endpoint Configuration
dbutils.widgets.text("embedding_endpoint", "simple_embedding", "Embedding Endpoint")
dbutils.widgets.text("llm_endpoint", "flan_t5_base_model", "LLM Endpoint")
dbutils.widgets.text("rag_endpoint", "appliance_chatbot", "RAG Endpoint")
dbutils.widgets.text("vector_search_endpoint", "doc_vector_endpoint", "Vector Search Endpoint")

# Processing Configuration
dbutils.widgets.text("chunk_size", "1000", "Text Chunk Size")
dbutils.widgets.text("chunk_overlap", "200", "Text Chunk Overlap")
dbutils.widgets.text("batch_size", "32", "Processing Batch Size")

# Index configuration
dbutils.widgets.text("index_name", "doc_idx", "Index name")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

## Cargar variables de configuración
Obtiene los valores de los widgets y construye los nombres completos de tablas, modelos y endpoints.

In [0]:
# Obtener valores de los widgets
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
ENVIRONMENT = dbutils.widgets.get("environment")

# Table names
DOCS_TEXT_TABLE = dbutils.widgets.get("docs_text_table")
DOCS_TRACK_TABLE = dbutils.widgets.get("docs_track_table")
EMBEDDINGS_TABLE = dbutils.widgets.get("embeddings_table")

# Volume path
PDF_VOLUME_PATH = dbutils.widgets.get("pdf_volume_path")

# Model names
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model_name")
LLM_MODEL_NAME = dbutils.widgets.get("llm_model_name")
RAG_MODEL_NAME = dbutils.widgets.get("rag_model_name")

# Endpoint names
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
RAG_ENDPOINT = dbutils.widgets.get("rag_endpoint")
VECTOR_SEARCH_ENDPOINT = dbutils.widgets.get("vector_search_endpoint")

# Processing parameters
CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

# Index configuration
INDEX = dbutils.widgets.get("index_name")

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Construir nombres completos
DOCS_TEXT_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DOCS_TEXT_TABLE}"
DOCS_TRACK_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DOCS_TRACK_TABLE}"
EMBEDDINGS_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{EMBEDDINGS_TABLE}"

EMBEDDING_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{EMBEDDING_MODEL_NAME}"
LLM_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{LLM_MODEL_NAME}"
RAG_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{RAG_MODEL_NAME}"

INDEX_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX}"

print("¡Configuración cargada correctamente!")
print(f"Environment: {ENVIRONMENT}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")

## Configurar experimento MLflow
Configura el experimento MLflow y lo crea si no existe.

In [0]:
# Configurar MLflow
mlflow.set_registry_uri("databricks-uc")

try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
        print(f"Creado nuevo experimento: {EXPERIMENT_NAME}")
    else:
        experiment_id = experiment.experiment_id
        print(f"Usando experimento existente: {EXPERIMENT_NAME}")
    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception as e:
    print(f"Error configurando experimento: {e}")
    mlflow.set_experiment("/Shared/RAG_Default")

In [0]:
# Parar todas las runs de MLflow activas
active_runs = mlflow.search_runs(filter_string="status = 'RUNNING'")
for run_id in active_runs['run_id']:
    mlflow.end_run(run_id)

## Funciones utilitarias
Funciones compartidas para logging, creación de tablas y utilidades de endpoints.

In [0]:
# Funciones utilitarias

def log_step(step_name, status="started", details=None):
    """Log pipeline step information"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    message = f"[{timestamp}] Step: {step_name} - Status: {status}"
    if details:
        message += f" - Details: {details}"
    print(message)
    if mlflow.active_run():
        mlflow.log_param(f"step_{step_name}_status", status)
        if details:
            mlflow.log_param(f"step_{step_name}_details", str(details))

def create_table_if_not_exists(table_name, schema_sql):
    """Create table if it doesn't exist"""
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} {schema_sql}")
        log_step("create_table", "success", f"Table {table_name} created/verified")
        return True
    except Exception as e:
        log_step("create_table", "failed", f"Error creating {table_name}: {e}")
        return False

def get_databricks_host():
    """Get Databricks workspace URL"""
    try:
        return spark.conf.get("spark.databricks.workspaceUrl")
    except:
        return "https://dbc-ad7d5e59-0280.cloud.databricks.com/"

def build_endpoint_url(endpoint_name):
    """Build serving endpoint URL"""
    host = get_databricks_host()
    return f"https://{host}/serving-endpoints/{endpoint_name}/invocations"

## Mostrar resumen de configuración
Imprime un resumen de la configuración actual del pipeline RAG.

In [0]:
print("="*60)
print("RAG DATABRICKS BLUETAB - CONFIGURATION SUMMARY")
print("="*60)
print(f"Environment: {ENVIRONMENT}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")
print()
print("TABLES:")
print(f"  - Documents Text: {DOCS_TEXT_TABLE_FULL}")
print(f"  - Documents Track: {DOCS_TRACK_TABLE_FULL}")
print(f"  - Embeddings: {EMBEDDINGS_TABLE_FULL}")
print()
print("MODELS:")
print(f"  - Embedding: {EMBEDDING_MODEL_FULL}")
print(f"  - LLM: {LLM_MODEL_FULL}")
print(f"  - RAG: {RAG_MODEL_FULL}")
print()
print("ENDPOINTS:")
print(f"  - Embedding: {EMBEDDING_ENDPOINT}")
print(f"  - LLM: {LLM_ENDPOINT}")
print(f"  - RAG: {RAG_ENDPOINT}")
print(f"  - Vector Search: {VECTOR_SEARCH_ENDPOINT}")
print()
print("PROCESSING:")
print(f"  - Chunk Size: {CHUNK_SIZE}")
print(f"  - Chunk Overlap: {CHUNK_OVERLAP}")
print(f"  - Batch Size: {BATCH_SIZE}")
print()
print("MLFLOW:")
print(f"  - Experiment: {EXPERIMENT_NAME}")
print("="*60)